# ZynNova — Free-form Unstructured Tet4 Meshing

This notebook tests the **new geometry-general meshing path**. It is intentionally different from the voxel/Hex8 and six-Tet4-per-voxel compatibility paths:

- reads the supplied `TetMesh-cell_nmc_grp.mphtxt` as a reference Tet4 mesh;
- extracts its edge-length and tetra-volume distribution;
- constructs a **non-box, wavy outer body** with two internal material inclusions and one true void;
- assembles one multi-domain PLC from arbitrary closed triangular shells;
- uses the compiled TetGen 1.6 C++ kernel to generate a conforming, spatially graded, fully unstructured Tet4 mesh;
- transfers reference mesh scale to the new geometry without copying its rectangular geometry;
- exports COMSOL MPHTXT, VTK, Gmsh MSH and Abaqus INP.

The production path never silently falls back to structured voxels.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import zynnova
from zynnova.geometry import TriangleMesh, tetra_quality
from zynnova.zynmorph import (
    FreeformRegion,
    LocalRefinementZone,
    SurfaceShell,
    TetGenMeshingConfig,
    assemble_freeform_plc,
    export_fem_mesh,
    mesh_freeform_geometry,
    mesh_freeform_like_reference,
    profile_reference_mesh,
    tetgen_config_from_reference,
    tetgen_native_diagnostics,
    tetgen_native_status,
)

print("ZynNova:", zynnova.__version__)
RUN_TETGEN = os.environ.get("ZYNNOVA_FREEFORM_RUN_TETGEN", "1") != "0"
OUTPUT_DIR = Path("outputs/freeform_unstructured")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("RUN_TETGEN =", RUN_TETGEN)


## 1. Inspect the supplied COMSOL Tet4 reference

The new reader supports both COMSOL native-text styles used by the old and current writers. The reference is used only as a **mesh sizing/style reference**; its cuboid shape is not imposed on the new geometry.

In [ ]:
reference_env = os.environ.get("ZYNNOVA_REFERENCE_MPHTXT")
reference_candidates = [
    *([Path(reference_env)] if reference_env else []),
    Path("TetMesh-cell_nmc_grp.mphtxt"),
    Path.cwd() / "TetMesh-cell_nmc_grp.mphtxt",
]
REFERENCE = next((p for p in reference_candidates if p.is_file()), None)

if REFERENCE is None:
    print("Reference MPHTXT not found. Reference-style sizing will be skipped.")
    reference_profile = None
else:
    reference_profile = profile_reference_mesh(REFERENCE)
    print("Reference:", REFERENCE.resolve())
    print("nodes      =", reference_profile.nodes)
    print("tetrahedra =", reference_profile.tetrahedra)
    print("regions    =", reference_profile.regions)
    print("extent [um] =", (
        (np.array(reference_profile.bbox_max_m_xyz) - np.array(reference_profile.bbox_min_m_xyz)) * 1e6
    ))
    print("edge quantiles [um] =", np.array(reference_profile.edge_length_quantiles_m) * 1e6)
    print("volume quantiles [um^3] =", np.array(reference_profile.tetra_volume_quantiles_m3) * 1e18)
    for region, rp in reference_profile.region_profiles.items():
        print(
            f"region {region}: tets={rp.tetrahedra}, "
            f"median edge={rp.edge_length_quantiles_m[3]*1e6:.3f} um, "
            f"q95 volume={rp.tetra_volume_quantiles_m3[5]*1e18:.3f} um^3"
        )


## 2. Build arbitrary non-box closed surfaces

The helper below creates triangulated wavy ellipsoids. It is just a compact test geometry; in production, the same `SurfaceShell` API can take STL/OBJ/PLY/NPZ surfaces from tomography, image reconstruction, CAD triangulation or external geometry tools.

In [ ]:
def wavy_ellipsoid(
    center_um,
    radii_um,
    *,
    n_lat=24,
    n_lon=48,
    waviness=0.0,
    phase=0.0,
):
    center = np.asarray(center_um, float) * 1e-6
    radii = np.asarray(radii_um, float) * 1e-6
    vertices = [center + np.array([0.0, 0.0, radii[2]])]

    for i in range(1, n_lat):
        theta = np.pi * i / n_lat
        for j in range(n_lon):
            phi = 2.0 * np.pi * j / n_lon
            modulation = 1.0 + waviness * (
                0.55*np.sin(3*phi + phase)*np.sin(theta)**2
                + 0.30*np.cos(5*phi - 1.7*theta)
                + 0.15*np.sin(4*theta + phase)
            )
            direction = np.array([
                np.sin(theta)*np.cos(phi),
                np.sin(theta)*np.sin(phi),
                np.cos(theta),
            ])
            vertices.append(center + radii * direction * modulation)

    south = len(vertices)
    vertices.append(center + np.array([0.0, 0.0, -radii[2]]))
    faces = []
    first_ring = 1
    for j in range(n_lon):
        faces.append([0, first_ring + j, first_ring + (j+1) % n_lon])

    for i in range(n_lat - 2):
        ring0 = 1 + i*n_lon
        ring1 = ring0 + n_lon
        for j in range(n_lon):
            a = ring0 + j
            b = ring0 + (j+1) % n_lon
            c = ring1 + (j+1) % n_lon
            d = ring1 + j
            if (i+j) % 2:
                faces.extend(([a,b,d], [b,c,d]))
            else:
                faces.extend(([a,b,c], [a,c,d]))

    last_ring = 1 + (n_lat-2)*n_lon
    for j in range(n_lon):
        faces.append([south, last_ring + (j+1) % n_lon, last_ring + j])
    return TriangleMesh(np.asarray(vertices), np.asarray(faces, np.int64))


outer = wavy_ellipsoid((50,25,25), (48,24,23), n_lat=28, n_lon=56, waviness=0.075, phase=0.2)
particle_a = wavy_ellipsoid((31,24,25), (11,8,9), n_lat=18, n_lon=36, waviness=0.08, phase=1.0)
particle_b = wavy_ellipsoid((72,28,20), (8,10,7), n_lat=18, n_lon=36, waviness=0.06, phase=2.2)
void_surface = wavy_ellipsoid((57,36,31), (5.0,3.8,4.3), n_lat=14, n_lon=28, waviness=0.05, phase=0.5)

shells = (
    SurfaceShell(outer, inside_region=1, outside_region=None, name="freeform_outer", maximum_triangle_area_m2=8.0e-12),
    SurfaceShell(particle_a, inside_region=2, outside_region=1, name="particle_A", maximum_triangle_area_m2=3.0e-12),
    SurfaceShell(particle_b, inside_region=3, outside_region=1, name="particle_B", maximum_triangle_area_m2=3.0e-12),
    SurfaceShell(void_surface, inside_region=99, outside_region=1, name="true_void", maximum_triangle_area_m2=2.0e-12),
)

plc = assemble_freeform_plc(shells)
print("PLC vertices  =", len(plc.vertices))
print("PLC triangles =", len(plc.triangles))
print("PLC regions   =", plc.regions)
print("markers       =", plc.marker_names)


### Visual check of the non-box PLC

In [ ]:
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")
for shell in shells:
    surf = shell.surface
    # Plot a sparse wireframe sample to keep the notebook responsive.
    sample = surf.faces[::max(1, len(surf.faces)//500)]
    for tri in surf.vertices[sample]:
        loop = np.vstack((tri, tri[0])) * 1e6
        ax.plot(loop[:,0], loop[:,1], loop[:,2], linewidth=0.45, alpha=0.55)
ax.set_xlabel("x [um]")
ax.set_ylabel("y [um]")
ax.set_zlabel("z [um]")
ax.set_title("Arbitrary outer shape + two material inclusions + one void")
ax.set_box_aspect((2,1,1))
plt.show()


## 3. Transfer the reference mesh scale and add local refinement

The reference MPHTXT has a broad edge/volume distribution. We map three new material regions onto three reference domains, then add a tighter local TetGen `-u` refinement zone near a high-curvature/interaction area.

In [ ]:
base_config = TetGenMeshingConfig(
    radius_edge_ratio=1.45,
    minimum_dihedral_degrees=8.0,
    optimization_level=2,
    local_refinement_zones=(
        LocalRefinementZone(
            center_m_xyz=(45e-6, 25e-6, 25e-6),
            radius_m=11e-6,
            maximum_tetra_volume_m3=2.0e-18,
            name="interaction_zone",
        ),
    ),
    consistency_check=True,
    conforming_delaunay=True,
    quiet=True,
)

if reference_profile is not None:
    config = tetgen_config_from_reference(
        reference_profile,
        # target -> reference domain
        region_map={1: 3, 2: 1, 3: 2},
        volume_quantile=0.95,
        linear_scale=0.85,
        base=base_config,
    )
else:
    config = TetGenMeshingConfig(
        **{
            **{name: getattr(base_config, name) for name in base_config.__dataclass_fields__},
            "phase_maximum_tetra_volume_m3": {1: 2.8e-17, 2: 1.8e-17, 3: 1.8e-17},
        }
    )

print("target per-region max tetra volumes [um^3]:")
for k, v in config.phase_maximum_tetra_volume_m3.items():
    print(" ", k, v*1e18)


## 4. Native TetGen runtime check

This must be available for the production path. The notebook does **not** substitute SciPy Delaunay or the structured voxel mesher.

In [ ]:
status = tetgen_native_status()
print(status)
if not status.available:
    print(json.dumps(tetgen_native_diagnostics(), indent=2, ensure_ascii=False, default=str))
if RUN_TETGEN and not status.available:
    raise RuntimeError(
        "TetGen native extension is required. Rebuild with "
        "python -m pip install -e \".[zynmorph-tetgen]\" -v and restart the kernel."
    )


## 5. Generate the fully unstructured multi-domain Tet4 mesh

In [ ]:
regions = (
    FreeformRegion(1, (50e-6, 8e-6, 25e-6), "matrix"),
    FreeformRegion(2, (31e-6, 24e-6, 25e-6), "particle_A"),
    FreeformRegion(3, (72e-6, 28e-6, 20e-6), "particle_B"),
)

fem = None
if RUN_TETGEN:
    if REFERENCE is not None:
        fem = mesh_freeform_like_reference(
            plc,
            regions,
            REFERENCE,
            region_map={1: 3, 2: 1, 3: 2},
            volume_quantile=0.95,
            linear_scale=0.85,
            tetgen_config=base_config,
            void_regions=(99,),
            holes_m_xyz=((57e-6, 36e-6, 31e-6),),
            maximum_tetrahedra=2_000_000,
        )
    else:
        fem = mesh_freeform_geometry(
            plc,
            regions,
            tetgen_config=config,
            void_regions=(99,),
            holes_m_xyz=((57e-6, 36e-6, 31e-6),),
            maximum_tetrahedra=2_000_000,
        )
    print("backend =", fem.backend)
    print("nodes   =", fem.mesh.n_nodes)
    print("Tet4    =", fem.mesh.n_cells)
    print("regions =", dict(zip(*np.unique(fem.mesh.cell_regions, return_counts=True))))
    print("quality =", fem.quality)
else:
    print("CI mode: native TetGen execution skipped; PLC/reference/API checks remain active.")


## 6. Prove that the result is nonuniform and not voxel-derived

In [ ]:
def tet_edge_lengths(mesh):
    pairs = ((0,1),(0,2),(0,3),(1,2),(1,3),(2,3))
    return np.concatenate([
        np.linalg.norm(mesh.nodes[mesh.tetrahedra[:,i]] - mesh.nodes[mesh.tetrahedra[:,j]], axis=1)
        for i,j in pairs
    ])

def tet_volumes(mesh):
    p = mesh.nodes[mesh.tetrahedra]
    return np.abs(np.einsum(
        "ij,ij->i", p[:,1]-p[:,0], np.cross(p[:,2]-p[:,0], p[:,3]-p[:,0])
    ))/6.0

if fem is not None:
    edges = tet_edge_lengths(fem.mesh)
    volumes = tet_volumes(fem.mesh)
    eq = np.quantile(edges, [0.01,0.05,0.5,0.95,0.99])*1e6
    vq = np.quantile(volumes, [0.01,0.05,0.5,0.95,0.99])*1e18
    print("edge q01/q05/q50/q95/q99 [um] =", eq)
    print("vol  q01/q05/q50/q95/q99 [um^3] =", vq)
    print("q95/q05 edge ratio =", eq[3]/eq[1])
    print("q95/q05 volume ratio =", vq[3]/vq[1])
    assert fem.quality.fem_ready
    assert fem.quality.inverted_cells == 0
    assert fem.quality.degenerate_cells == 0
    assert eq[3] / eq[1] > 1.25, "mesh is unexpectedly uniform"
    assert vq[3] / vq[1] > 2.0, "tetra volume grading is unexpectedly weak"
    assert not fem.mesh.metadata.get("rectangular_domain_assumed", True)


## 7. Visualize a random subset of Tet4 edges

In [ ]:
if fem is not None:
    rng = np.random.default_rng(20260818)
    ids = rng.choice(fem.mesh.n_cells, size=min(1600, fem.mesh.n_cells), replace=False)
    fig = plt.figure(figsize=(11, 7))
    ax = fig.add_subplot(111, projection="3d")
    pairs = ((0,1),(0,2),(0,3),(1,2),(1,3),(2,3))
    for tet in fem.mesh.tetrahedra[ids]:
        xyz = fem.mesh.nodes[tet]*1e6
        for i,j in pairs:
            ax.plot(
                [xyz[i,0],xyz[j,0]],
                [xyz[i,1],xyz[j,1]],
                [xyz[i,2],xyz[j,2]],
                linewidth=0.22,
                alpha=0.16,
            )
    ax.set_xlabel("x [um]")
    ax.set_ylabel("y [um]")
    ax.set_zlabel("z [um]")
    ax.set_title("Sample of free-form spatially graded TetGen Tet4 cells")
    ax.set_box_aspect((2,1,1))
    plt.show()


## 8. Export COMSOL MPHTXT and common FEM formats

In [ ]:
if fem is not None:
    exported = export_fem_mesh(
        fem,
        OUTPUT_DIR,
        formats=("mphtxt", "vtk", "msh", "inp"),
        export_boundary=True,
        comsol_options={
            "include_default_battery_selections": False,
            "include_default_boundary_unions": False,
            "include_coordinate_boundaries": False,
            "verify": True,
        },
    )
    for key, path in exported.exports.items():
        print(f"{key:14s}", path, path.stat().st_size)


## 9. Save a validation summary

In [ ]:
summary = {
    "reference": None if reference_profile is None else {
        "nodes": reference_profile.nodes,
        "tetrahedra": reference_profile.tetrahedra,
        "regions": reference_profile.regions,
        "edge_quantiles_um": (np.array(reference_profile.edge_length_quantiles_m)*1e6).tolist(),
        "tetra_volume_quantiles_um3": (np.array(reference_profile.tetra_volume_quantiles_m3)*1e18).tolist(),
    },
    "plc": {
        "vertices": len(plc.vertices),
        "triangles": len(plc.triangles),
        "regions": plc.regions,
        "markers": plc.marker_names,
    },
    "native_status": {
        "available": status.available,
        "version": status.version,
        "reason": status.reason,
    },
    "mesh": None,
}
if fem is not None:
    edges = tet_edge_lengths(fem.mesh)
    volumes = tet_volumes(fem.mesh)
    summary["mesh"] = {
        "backend": fem.backend,
        "nodes": fem.mesh.n_nodes,
        "tetrahedra": fem.mesh.n_cells,
        "regions": {str(int(r)): int(np.count_nonzero(fem.mesh.cell_regions==r)) for r in np.unique(fem.mesh.cell_regions)},
        "edge_quantiles_um": (np.quantile(edges,[0.01,0.05,0.5,0.95,0.99])*1e6).tolist(),
        "tetra_volume_quantiles_um3": (np.quantile(volumes,[0.01,0.05,0.5,0.95,0.99])*1e18).tolist(),
        "quality": {
            "inverted": fem.quality.inverted_cells,
            "degenerate": fem.quality.degenerate_cells,
            "minimum_mean_ratio": fem.quality.minimum_mean_ratio,
            "median_mean_ratio": fem.quality.median_mean_ratio,
            "fem_ready": fem.quality.fem_ready,
        },
    }

summary_path = OUTPUT_DIR / "freeform_validation.json"
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False, default=str)+"\n", encoding="utf-8")
print(summary_path)
print(json.dumps(summary, indent=2, ensure_ascii=False, default=str)[:8000])
